# XGBoost

## 1. What is XGBoost?

**XGBoost (Extreme Gradient Boosting)** is an optimized and regularized implementation of **Gradient Boosting** that uses decision trees as weak learners.

The main idea is:

> **Build trees sequentially, where each new tree tries to improve the predictions made by the previous trees.**

### Random Forest vs XGBoost

| Random Forest | XGBoost |
|---|---|
| Bagging | Boosting |
| Trees are mostly independent | Trees are built sequentially |
| Uses random samples of rows | Each tree focuses on reducing previous model's loss |
| Random feature selection | Gradient-based optimization |
| Mainly reduces variance | Mainly reduces bias |

---

## 2. Basic Working

Suppose we have a model that makes an initial prediction.

Let the initial prediction be:

$$
\hat{y}^{(0)}
$$

The first tree produces a correction:

$$
f_1(X)
$$

The updated prediction becomes:

$$
\hat{y}^{(1)}
=
\hat{y}^{(0)} + \eta f_1(X)
$$

Then another tree is built to improve the current model:

$$
\hat{y}^{(2)}
=
\hat{y}^{(1)} + \eta f_2(X)
$$

This continues for multiple trees.

After \(M\) trees:

$$
\boxed{
\hat{y}^{(M)}
=
\hat{y}^{(0)}
+
\eta
\sum_{m=1}^{M} f_m(X)
}
$$

### Meaning of the symbols

- \(M\) = number of trees
- \(f_m(X)\) = prediction/correction produced by tree \(m\)
- \(\eta\) = learning rate
- \(\hat{y}^{(0)}\) = initial prediction
- \(\hat{y}^{(M)}\) = final prediction after all trees

---

## 3. Why is it called "Gradient" Boosting?

The new tree is not simply trained randomly.

It is trained according to the **gradient of the loss function**.

If our loss is:

$$
L(y,\hat{y})
$$

we calculate how the loss changes with respect to the current prediction:

$$
\frac{\partial L}{\partial \hat{y}}
$$

The next tree tries to make a correction that **reduces the loss**.

So the overall idea is:

$$
\boxed{
\text{Current Model}
+
\text{Small Loss-Reducing Correction}
}
$$

Repeated many times.

---

## 4. Simple Example

Suppose the actual house prices are:

$$
y = [100,200,300]
$$

Our first model predicts:

$$
\hat{y}^{(0)} = [120,180,250]
$$

The predictions are not perfect.

For squared-error loss, the residuals are:

$$
y-\hat{y}^{(0)}
=
[-20,20,50]
$$

The next tree tries to learn a correction related to these errors.

If the learning rate is:

$$
\eta = 0.1
$$

the correction contributed by the tree is scaled down:

$$
0.1 \times f_1(X)
$$

This prevents each individual tree from making an excessively large change to the model.

---

### Core idea to remember

$$
\boxed{
\text{XGBoost}
=
\text{Sequential Decision Trees}
+
\text{Gradient Optimization}
+
\text{Regularization}
}
$$

The **gradient optimization** part is what we need to understand first before going into the more specific XGBoost mathematics.

# XGBoost — 2. Objective Function

The **objective function** tells XGBoost what it should minimize during training.

It has two parts:

1. **Training loss** — measures how wrong the predictions are.
2. **Regularization** — controls the complexity of the trees.

The general XGBoost objective function is:

$$\text{Objective}=\text{Training Loss}+\text{Regularization}$$

More formally, at boosting step $t$:

$$\mathcal{L}^{(t)}=\sum_{i=1}^{n}l(y_i,\hat{y}_i^{(t)})+\Omega(f_t)$$

where:

- $n$ = number of training samples
- $y_i$ = actual value for sample $i$
- $\hat{y}_i^{(t)}$ = prediction after adding the $t$-th tree
- $l$ = loss function
- $f_t$ = the new decision tree
- $\Omega(f_t)$ = regularization term for the new tree

## 1. Training Loss

The loss measures how far predictions are from the actual values.

For regression, XGBoost can use squared error:

$$l(y_i,\hat{y}_i)=\frac{1}{2}(y_i-\hat{y}_i)^2$$

A smaller loss means the predictions are closer to the actual values.

## 2. Why Regularization?

Suppose two trees produce almost the same prediction quality:

- Tree A is simple.
- Tree B is extremely deep with many leaves.

XGBoost prefers the **simpler tree** because a complicated tree can overfit.

Therefore:

$$\text{Objective}=\text{How wrong?}+\text{How complex?}$$

The model tries to find a tree that gives the **lowest total objective**.

### Key idea

XGBoost does **not** simply ask:

> "Which tree predicts the training data best?"

It asks:

> **"Which tree improves the loss while keeping the model sufficiently simple?"**

The regularization part $\Omega(f_t)$ is what we will examine in detail later.

# XGBoost — 3. Regularization

Regularization is used in XGBoost to **control the complexity of each decision tree** and reduce overfitting.

The regularization term is:

$$\Omega(f_t)=\gamma T+\frac{1}{2}\lambda\sum_{j=1}^{T}w_j^2$$

where:

- $T$ = number of leaves in the tree
- $w_j$ = prediction value of leaf $j$
- $\gamma$ = penalty for creating additional leaves
- $\lambda$ = penalty on large leaf weights

So the objective becomes:

$$\mathcal{L}^{(t)}=\sum_{i=1}^{n}l(y_i,\hat{y}_i^{(t)})+\gamma T+\frac{1}{2}\lambda\sum_{j=1}^{T}w_j^2$$

## 1. Leaf Penalty — $\gamma$

The term $\gamma T$ penalizes trees with many leaves.

If a tree creates more leaves:

$$T\uparrow \quad\Rightarrow\quad \gamma T\uparrow$$

Therefore, XGBoost will only create an additional split when the improvement in the objective is large enough to justify the extra complexity.

Higher $\gamma$ → fewer splits → simpler trees.

## 2. Weight Penalty — $\lambda$

The second term penalizes large leaf values:

$$\frac{1}{2}\lambda\sum_{j=1}^{T}w_j^2$$

If leaf weights become very large, the penalty increases.

Higher $\lambda$ → smaller leaf weights → more conservative predictions.

## Main idea

XGBoost balances:

$$\text{Good Predictions}+\text{Low Model Complexity}$$

This is one of the major reasons XGBoost is resistant to overfitting compared with an unrestricted boosting model.

# XGBoost — 4. Gradient and Hessian

XGBoost uses **two pieces of information** from the loss function:

1. **Gradient** — first derivative
2. **Hessian** — second derivative

These allow XGBoost to determine how the new tree should improve the current predictions.

## 1. Gradient

The gradient tells us the **direction and magnitude of the error** with respect to the prediction.

$$g_i=\frac{\partial l(y_i,\hat{y}_i)}{\partial \hat{y}_i}$$

For each training sample, XGBoost calculates one gradient value.

A larger magnitude means the current prediction needs a stronger correction.

---

## 2. Hessian

The Hessian is the **second derivative** of the loss.

$$h_i=\frac{\partial^2 l(y_i,\hat{y}_i)}{\partial \hat{y}_i^2}$$

It tells XGBoost how the gradient itself is changing.

So:

- **Gradient** → direction of correction
- **Hessian** → how the loss behaves around that prediction

---

## 3. Why does XGBoost use both?

Basic Gradient Boosting mainly uses the gradient.

XGBoost uses:

$$\text{Gradient}+\text{Hessian}$$

This is called **second-order optimization**.

It gives XGBoost more information about the shape of the loss function and allows it to calculate better tree updates.

For each training sample, XGBoost therefore calculates:

$$\boxed{g_i,\ h_i}$$

These values are then used to determine the optimal leaf weights and whether a tree split provides enough improvement.

### Important distinction

Don't think of the Hessian as another type of error.

- Gradient tells **which direction and how strongly to correct**.
- Hessian tells **how the loss curve is changing around that point**.

This gradient + Hessian calculation is the mathematical foundation behind XGBoost's tree construction.

Yes — the missing piece is **how the Hessian actually enters the calculation**. Let's ignore the complicated parts and trace exactly what XGBoost does with the second derivative.

# How XGBoost actually uses the second derivative

Suppose the current prediction for one sample is:

$$\hat{y}=80$$

and the actual value is:

$$y=100$$

XGBoost wants to decide:

> **How much should the new tree change the prediction?**

---

## 1. First, calculate the gradient

For squared-error loss:

$$L=\frac{1}{2}(y-\hat{y})^2$$

The first derivative is:

$$g=\frac{\partial L}{\partial\hat{y}}=\hat{y}-y$$

So:

$$g=80-100=-20$$

The gradient says:

> "The prediction needs to move upward."

But **how much should we move?**

This is where the second derivative comes in.

---

## 2. Calculate the second derivative

The second derivative is:

$$h=\frac{\partial^2L}{\partial\hat{y}^2}=1$$

Now XGBoost has:

$$g=-20$$

and:

$$h=1$$

The Hessian tells XGBoost about the **curvature of the loss around the current prediction**.

But how does it actually use that information?

---

# 3. XGBoost approximates the loss

This is the key step.

Instead of directly trying every possible new prediction, XGBoost approximates what will happen to the loss if we change the prediction by some amount $w$.

Using a second-order Taylor approximation:

$$L(\hat{y}+w)\approx L(\hat{y})+gw+\frac{1}{2}hw^2$$

Look carefully at this equation.

There are three parts:

$$L(\hat{y})$$

Current loss.

$$gw$$

Effect predicted using the **first derivative**.

$$\frac{1}{2}hw^2$$

Effect predicted using the **second derivative**.

This is exactly where the Hessian is being used.

---

# 4. Let's put numbers into it

We had:

$$g=-20$$

and:

$$h=1$$

Therefore:

$$L(\hat{y}+w)\approx L(\hat{y})-20w+\frac{1}{2}w^2$$

Now XGBoost asks:

> What value of $w$ gives us the lowest approximate loss?

We differentiate this with respect to $w$:

$$\frac{dL}{dw}=-20+w$$

Set it equal to zero:

$$-20+w=0$$

Therefore:

$$w=20$$

So the model says:

> **Increase the prediction by approximately 20.**

Our prediction changes from:

$$80\rightarrow100$$

---

# 5. So what did the second derivative actually do?

This is the important part.

Without the second derivative, we would only know:

$$g=-20$$

which tells us the **direction** of movement.

But with the second derivative:

$$h=1$$

XGBoost can estimate the **curvature of the loss** and determine how large the correction should be.

The optimal correction comes from:

$$w^*=-\frac{g}{h}$$

For our example:

$$w^*=-\frac{-20}{1}=20$$

So:

**Gradient:**

$$g=-20$$

→ tells us to move upward.

**Hessian:**

$$h=1$$

→ tells us the curvature/scaling of that correction.

Together:

$$\text{Correction}=-\frac{g}{h}$$

---

# 6. Now why is this useful for XGBoost?

Here's where XGBoost becomes more interesting.

A tree leaf usually contains **many samples**, not just one.

Suppose a leaf contains three samples:

$$g_1=-20,\quad g_2=-10,\quad g_3=5$$

and:

$$h_1=1,\quad h_2=1,\quad h_3=1$$

XGBoost sums them:

$$G=\sum_i g_i=-25$$

$$H=\sum_i h_i=3$$

Then the optimal leaf weight is:

$$w^*=-\frac{G}{H+\lambda}$$

If:

$$\lambda=2$$

then:

$$w^*=-\frac{-25}{3+2}=5$$

So the **second derivative directly appears in the denominator**.

It affects the size of the correction.

---

# 7. The easiest way to remember it

Think of the optimization like this:

**First derivative:**

> "Which direction should I move?"

**Second derivative:**

> "How does the loss curve behave as I move?"

Then XGBoost combines both to calculate:

> **"What is the best size of the correction?"**

So the Hessian isn't just calculated and ignored.

It directly affects the **leaf weight**:

$$w^*=-\frac{G}{H+\lambda}$$

And because every leaf gets its weight from this calculation, the Hessian directly influences what the new tree contributes to the model.

### The complete chain

$$\text{Loss}\rightarrow\text{Gradient + Hessian}\rightarrow\text{Optimal Leaf Weight}\rightarrow\text{New Tree}\rightarrow\text{Updated Prediction}$$

**That is exactly how XGBoost uses the second derivative.**

# XGBoost — 5. Classification

XGBoost can be used for **classification** by building a sequence of decision trees that gradually improves the model's class predictions.

## 1. Binary Classification

Suppose we want to predict whether an observation belongs to:

- Class $0$
- Class $1$

Unlike regression, the final output should represent a **probability**.

For example:

$$P(y=1\mid X)=0.82$$

This means the model estimates an $82\%$ probability that the sample belongs to class $1$.

---

## 2. Why can't the tree directly output probability?

The individual XGBoost trees produce numerical values called **leaf weights**.

The model first combines these values:

$$F(X)=\sum_{m=1}^{M}\eta f_m(X)$$

This $F(X)$ is called the **raw score** or **log-odds score**.

It is not yet a probability.

---

## 3. Convert raw score to probability

XGBoost uses the **sigmoid function** for binary classification:

$$p=\frac{1}{1+e^{-F(X)}}$$

This converts any raw score into a value between $0$ and $1$.

For example, if:

$$F(X)=2$$

then:

$$p=\frac{1}{1+e^{-2}}\approx0.881$$

So the model predicts approximately:

$$P(y=1\mid X)=0.881$$

or **88.1% probability of class 1**.

---

## 4. Convert probability into a class

Usually a threshold of $0.5$ is used.

If:

$$p\geq0.5$$

predict:

$$\hat{y}=1$$

Otherwise:

$$\hat{y}=0$$

So:

$$p=0.881\rightarrow\hat{y}=1$$

while:

$$p=0.32\rightarrow\hat{y}=0$$

---

## 5. Where do Gradient and Hessian come in?

This connects directly to what we just learned.

During training, XGBoost calculates the gradient and Hessian of the **classification loss** for every sample.

For binary classification, a commonly used loss is **log loss**.

The model uses the resulting gradients and Hessians to determine:

- which splits are useful
- the optimal value of each leaf
- how much the new tree should change the current predictions

So the overall process is:

$$X\rightarrow\text{Trees}\rightarrow\text{Raw Score}\rightarrow\text{Sigmoid}\rightarrow\text{Probability}\rightarrow\text{Class}$$

### Important distinction

The **tree itself does not directly output the final class**.

It contributes a numerical correction to the model:

$$F^{(t)}(X)=F^{(t-1)}(X)+\eta f_t(X)$$

After all trees have been added, XGBoost converts the final raw score into a probability using the sigmoid function.

For **multiclass classification**, the idea is extended using multiple class scores and **softmax** rather than sigmoid.

# XGBoost — 6. Regression

XGBoost can also be used for **regression**, where the target is a continuous numerical value.

Examples:

- House price → $₹85$ lakh
- Temperature → $31.5^\circ C$
- Sales → $12500$

## 1. Basic idea

The model starts with an initial prediction and then sequentially adds trees that correct the current prediction.

$$\hat{y}^{(t)}=\hat{y}^{(t-1)}+\eta f_t(X)$$

After $M$ trees:

$$\hat{y}^{(M)}=\hat{y}^{(0)}+\eta\sum_{m=1}^{M}f_m(X)$$

Unlike classification, we don't convert the final output through sigmoid.

The final value itself is the prediction.

For example:

$$\hat{y}=₹82,50,000$$

means the predicted house price is ₹82.5 lakh.

---

## 2. Regression Loss

For regression, XGBoost can use squared error.

$$l(y_i,\hat{y}_i)=\frac{1}{2}(y_i-\hat{y}_i)^2$$

Its gradient is:

$$g_i=\hat{y}_i-y_i$$

And its Hessian is:

$$h_i=1$$

So the gradient and Hessian we studied earlier are directly used during regression tree construction.

---

## 3. Small Example

Suppose the actual values are:

$$y=[100,200,300]$$

and the current model predicts:

$$\hat{y}=[80,220,280]$$

The gradients are:

$$g=[-20,20,-20]$$

The Hessians are:

$$h=[1,1,1]$$

XGBoost uses these values to determine how the next tree should modify the predictions.

After adding the new tree:

$$\hat{y}^{(t)}=\hat{y}^{(t-1)}+\eta f_t(X)$$

The predictions should move closer to the actual values.

---

## Classification vs Regression

| | Classification | Regression |
|---|---|---|
| Output | Probability / class | Continuous value |
| Common output transformation | Sigmoid / Softmax | None |
| Example | Spam / Not Spam | House price |
| Loss | Log loss | Squared error |
| Gradient & Hessian | Used | Used |

The **core XGBoost mechanism remains the same** in both cases: calculate gradients and Hessians, construct trees using them, and sequentially improve the model.

Yes. Let's do a **complete 2-row XGBoost regression example from start to finish**, including:

**initial prediction → loss → gradient → second derivative → objective → leaf weight → new prediction.**

I’ll keep the numbers very small so you can calculate everything by hand.

# XGBoost Regression — 2 Row Complete Example

Suppose we have only 2 training samples:

| Row | Actual $y$ |
|---|---:|
| 1 | 10 |
| 2 | 20 |

We want to train one XGBoost tree.

For simplicity, assume the initial prediction for both rows is:

$$\hat{y}^{(0)}=15$$

So initially:

| Row | Actual | Initial prediction |
|---|---:|---:|
| 1 | 10 | 15 |
| 2 | 20 | 15 |

---

# Step 1 — Calculate the loss

We'll use squared-error loss:

$$l(y_i,\hat{y}_i)=\frac{1}{2}(y_i-\hat{y}_i)^2$$

### Row 1

Actual:

$$y_1=10$$

Prediction:

$$\hat{y}_1=15$$

Therefore:

$$l_1=\frac{1}{2}(10-15)^2=12.5$$

### Row 2

$$l_2=\frac{1}{2}(20-15)^2=12.5$$

Total loss:

$$L=12.5+12.5=25$$

---

# Step 2 — Calculate the Gradient

The gradient is:

$$g_i=\frac{\partial l(y_i,\hat{y}_i)}{\partial\hat{y}_i}=\hat{y}_i-y_i$$

### Row 1

$$g_1=15-10=5$$

### Row 2

$$g_2=15-20=-5$$

So:

| Row | Actual | Prediction | Gradient |
|---|---:|---:|---:|
| 1 | 10 | 15 | $5$ |
| 2 | 20 | 15 | $-5$ |

### What does this mean?

Row 1:

$$g_1=5$$

The prediction is **too high**, so we need to push it downward.

Row 2:

$$g_2=-5$$

The prediction is **too low**, so we need to push it upward.

---

# Step 3 — Calculate the Hessian

The Hessian is the second derivative:

$$h_i=\frac{\partial^2l(y_i,\hat{y}_i)}{\partial\hat{y}_i^2}$$

For squared-error loss:

$$h_i=1$$

Therefore:

$$h_1=1$$

$$h_2=1$$

Our table is now:

| Row | Actual | Prediction | Gradient | Hessian |
|---|---:|---:|---:|---:|
| 1 | 10 | 15 | $5$ | $1$ |
| 2 | 20 | 15 | $-5$ | $1$ |

---

# Step 4 — Build a tree

Now XGBoost wants to find a tree that can improve these predictions.

Suppose our tree makes **one leaf containing both rows**.

So:

$$I=\{1,2\}$$

XGBoost sums the gradients:

$$G=\sum_{i\in I}g_i$$

Therefore:

$$G=5+(-5)=0$$

And sums the Hessians:

$$H=\sum_{i\in I}h_i$$

Therefore:

$$H=1+1=2$$

So this leaf has:

$$G=0,\qquad H=2$$

---

# Step 5 — Calculate the leaf weight

The optimal leaf weight is:

$$w^*=-\frac{G}{H+\lambda}$$

For now, let's assume:

$$\lambda=0$$

Therefore:

$$w^*=-\frac{0}{2}=0$$

So the tree contributes:

$$0$$

That means **this tree cannot improve the prediction if both rows remain in the same leaf**.

This is actually an important result.

---

# Step 6 — Why does XGBoost want to split?

Look at our two rows again.

Row 1 needs:

$$-5$$

correction.

Row 2 needs:

$$+5$$

correction.

But they're currently in the **same leaf**.

One leaf can only have **one weight**.

Therefore, it cannot simultaneously give:

$$-5$$

to Row 1 and:

$$+5$$

to Row 2.

So XGBoost considers splitting the tree.

Suppose a feature allows us to separate them:

**Left leaf → Row 1**

**Right leaf → Row 2**

Now let's calculate the two leaf weights.

---

# Step 7 — Left leaf

Left leaf contains Row 1.

Therefore:

$$G_L=5$$

$$H_L=1$$

The leaf weight is:

$$w_L=-\frac{G_L}{H_L+\lambda}$$

With $\lambda=0$:

$$w_L=-\frac{5}{1}=-5$$

So Row 1 receives a correction of:

$$-5$$

Its new prediction becomes:

$$15+(-5)=10$$

Perfect!

---

# Step 8 — Right leaf

Right leaf contains Row 2.

Therefore:

$$G_R=-5$$

$$H_R=1$$

Its weight is:

$$w_R=-\frac{-5}{1}=5$$

So Row 2 receives:

$$+5$$

Its new prediction becomes:

$$15+5=20$$

Again, perfect!

---

# Step 9 — Apply learning rate

In real XGBoost, we normally don't allow the tree to make the entire correction immediately.

Suppose:

$$\eta=0.5$$

Then the actual correction is:

$$\eta w$$

### Row 1

$$0.5(-5)=-2.5$$

New prediction:

$$15-2.5=12.5$$

### Row 2

$$0.5(5)=2.5$$

New prediction:

$$15+2.5=17.5$$

So after one tree:

| Row | Actual | Before | Tree correction | After |
|---|---:|---:|---:|---:|
| 1 | 10 | 15 | $-2.5$ | $12.5$ |
| 2 | 20 | 15 | $+2.5$ | $17.5$ |

The predictions moved **toward the actual values**.

---

# Step 10 — Calculate the new loss

Before the tree:

$$L=25$$

After the tree:

### Row 1

$$l_1=\frac{1}{2}(10-12.5)^2=3.125$$

### Row 2

$$l_2=\frac{1}{2}(20-17.5)^2=3.125$$

Total:

$$L=3.125+3.125=6.25$$

So:

$$25\rightarrow6.25$$

The loss decreased significantly.

---

# Step 11 — Where does the Objective Function come in?

Remember:

$$\text{Objective}=\text{Loss}+\text{Regularization}$$

Suppose we use:

$$\gamma=1$$

and our tree has:

$$T=2$$

leaves.

The regularization term is:

$$\Omega(f)=\gamma T+\frac{1}{2}\lambda\sum_{j=1}^{T}w_j^2$$

Since we assumed:

$$\lambda=0$$

we get:

$$\Omega(f)=1\times2=2$$

Therefore the objective is:

$$\text{Objective}=6.25+2=8.25$$

XGBoost doesn't just want low loss.

It wants a **low objective**, meaning:

$$\text{Low Loss}+\text{Low Complexity}$$

---

# The complete XGBoost flow

This is the important part to remember:

$$\text{Initial Prediction}$$

↓

$$\text{Calculate Loss}$$

↓

$$\text{Calculate Gradient }g_i$$

↓

$$\text{Calculate Hessian }h_i$$

↓

$$\text{Try Tree Splits}$$

↓

$$\text{Calculate }G=\sum g_i\text{ and }H=\sum h_i$$

↓

$$\text{Calculate Leaf Weight }w^*=-\frac{G}{H+\lambda}$$

↓

$$\text{Calculate Objective}$$

↓

$$\text{Choose the useful split}$$

↓

$$\text{Apply Learning Rate}$$

↓

$$\text{Update Predictions}$$

---

## And this is the key thing you were asking about earlier

The **second derivative isn't calculated just for information**.

It actually enters here:

$$w^*=-\frac{G}{H+\lambda}$$

The Hessian $H$ is in the **denominator**.

Therefore, it directly affects the size of the leaf's correction.

That's how the second derivative actually becomes part of the model's prediction.

# XGBoost — Hyperparameters

XGBoost has several important hyperparameters that control **how many trees are built, how complex each tree can become, how strongly the model is regularized, and how much randomness is introduced**.

## 1. `n_estimators`

Controls the **number of trees** built sequentially.

```python
XGBRegressor(n_estimators=100)
```

means 100 boosting rounds.

$$\hat{y}^{(M)}=\hat{y}^{(0)}+\eta\sum_{m=1}^{M}f_m(X)$$

Here, $M$ corresponds to the number of trees.

More trees → more opportunities to correct errors.

Too many → possible overfitting and more computation.

---

## 2. `learning_rate`

Controls how much each new tree contributes to the model.

$$\hat{y}^{(t)}=\hat{y}^{(t-1)}+\eta f_t(X)$$

where $\eta$ is the learning rate.

Example:

```python
learning_rate=0.1
```

means only 10% of the new tree's contribution is added.

### Relationship with `n_estimators`

Small learning rate usually requires more trees.

$$\text{Small }\eta\rightarrow\text{More trees usually needed}$$

$$\text{Large }\eta\rightarrow\text{Fewer trees usually needed}$$

A common approach is to use a **smaller learning rate with more trees**.

---

## 3. `max_depth`

Controls the **maximum depth of each decision tree**.

```python
max_depth=3
```

means each tree can grow to a maximum depth of 3.

Smaller depth:

$$\text{Simpler trees}\rightarrow\text{lower overfitting risk}$$

Larger depth:

$$\text{More complex trees}\rightarrow\text{higher overfitting risk}$$

XGBoost commonly uses relatively shallow trees because many trees are combined together.

---

## 4. `min_child_weight`

Controls the **minimum sum of Hessians required in a child node** before XGBoost allows a split.

This is different from Random Forest's `min_samples_leaf`.

Remember that XGBoost calculates:

$$H=\sum_{i\in I}h_i$$

For a proposed child node, XGBoost checks whether its Hessian sum is large enough.

Higher `min_child_weight`:

$$\text{Harder to create new leaves}\rightarrow\text{Simpler tree}$$

Lower `min_child_weight`:

$$\text{Easier to create leaves}\rightarrow\text{More complex tree}$$

This helps control overfitting.

---

## 5. `subsample`

Controls the **fraction of training rows used for each tree**.

Example:

```python
subsample=0.8
```

means approximately 80% of the training rows are randomly selected for each boosting round.

If there are 1000 rows:

$$0.8\times1000=800$$

approximately 800 rows are used for a tree.

Lower values introduce more randomness and can reduce overfitting.

---

## 6. `colsample_bytree`

Controls the **fraction of features randomly selected for each tree**.

Example:

```python
colsample_bytree=0.8
```

If there are 10 features:

$$0.8\times10=8$$

approximately 8 features are available to that tree.

This introduces feature randomness and can help reduce overfitting.

---

## 7. `gamma`

Controls the **minimum loss reduction required to make a split**.

Remember the regularization term:

$$\Omega(f_t)=\gamma T+\frac{1}{2}\lambda\sum_{j=1}^{T}w_j^2$$

A split should provide enough improvement to justify increasing the tree complexity.

Higher `gamma`:

$$\text{More improvement required}\rightarrow\text{Fewer splits}$$

Lower `gamma`:

$$\text{Less improvement required}\rightarrow\text{More splits}$$

So `gamma` directly controls how conservative XGBoost is about creating new branches.

---

## 8. `reg_lambda`

`reg_lambda` is the **L2 regularization parameter** $\lambda$.

It penalizes large leaf weights.

The leaf weight formula is:

$$w^*=-\frac{G}{H+\lambda}$$

Increasing $\lambda$ increases the denominator:

$$\lambda\uparrow\rightarrow H+\lambda\uparrow\rightarrow |w^*|\downarrow$$

Therefore, the leaf corrections become smaller.

Higher `reg_lambda` → stronger L2 regularization → more conservative model.

---

## 9. `reg_alpha`

`reg_alpha` is the **L1 regularization parameter** $\alpha$.

It also penalizes large leaf weights, but unlike L2 regularization, L1 can push some weights toward exactly zero.

Conceptually:

$$\text{L1}\rightarrow\text{encourages sparse leaf weights}$$

Higher `reg_alpha` → stronger regularization → potentially simpler model.

---

# Quick Comparison

| Hyperparameter | Controls |
|---|---|
| `n_estimators` | Number of trees |
| `learning_rate` | Contribution of each tree |
| `max_depth` | Maximum tree depth |
| `min_child_weight` | Minimum Hessian weight needed for a child |
| `subsample` | Fraction of rows used per tree |
| `colsample_bytree` | Fraction of features used per tree |
| `gamma` | Minimum improvement required for a split |
| `reg_lambda` | L2 regularization |
| `reg_alpha` | L1 regularization |

### The important relationships

$$\text{n\_estimators}\rightarrow\text{How many correction steps}$$

$$\text{learning\_rate}\rightarrow\text{How large each correction is}$$

$$\text{max\_depth}\rightarrow\text{How complex each tree can become}$$

$$\text{min\_child\_weight}\rightarrow\text{How easily nodes can split}$$

$$\text{subsample}\rightarrow\text{How many rows each tree sees}$$

$$\text{colsample\_bytree}\rightarrow\text{How many features each tree sees}$$

$$\text{gamma}\rightarrow\text{How much improvement a split must provide}$$

$$\text{reg\_lambda}\rightarrow\text{L2 penalty on leaf weights}$$

$$\text{reg\_alpha}\rightarrow\text{L1 penalty on leaf weights}$$

# Entire XGBoost Flow

## 1. Our training data

Suppose we have:

| Row | Feature $X$ | Actual $y$ |
|---|---:|---:|
| 1 | 1 | 10 |
| 2 | 2 | 12 |
| 3 | 3 | 20 |
| 4 | 4 | 22 |

We want XGBoost to predict $y$ from $X$.

---

# 2. Start with an initial prediction

Before building any tree, suppose XGBoost starts with:

$$\hat{y}^{(0)}=16$$

for every row.

So:

| Row | Actual $y$ | Initial prediction $\hat{y}$ |
|---|---:|---:|
| 1 | 10 | 16 |
| 2 | 12 | 16 |
| 3 | 20 | 16 |
| 4 | 22 | 16 |

The model is obviously not perfect.

Now XGBoost asks:

> **How should the first tree correct these predictions?**

This is where the next sections connect.

---

# 3. Calculate the loss

For regression, use squared-error loss:

$$l_i=\frac{1}{2}(y_i-\hat{y}_i)^2$$

For row 1:

$$l_1=\frac{1}{2}(10-16)^2=18$$

For row 2:

$$l_2=\frac{1}{2}(12-16)^2=8$$

For row 3:

$$l_3=\frac{1}{2}(20-16)^2=8$$

For row 4:

$$l_4=\frac{1}{2}(22-16)^2=18$$

Total loss:

$$L=18+8+8+18=52$$

So the current model has loss $52$.

---

# 4. Calculate gradient

For squared-error loss:

$$g_i=\hat{y}_i-y_i$$

Therefore:

| Row | $y$ | $\hat{y}$ | Gradient $g$ |
|---|---:|---:|---:|
| 1 | 10 | 16 | 6 |
| 2 | 12 | 16 | 4 |
| 3 | 20 | 16 | -4 |
| 4 | 22 | 16 | -6 |

### What does this tell us?

Positive gradient:

```text
Prediction is too HIGH
        ↓
Need negative correction
```

Negative gradient:

```text
Prediction is too LOW
        ↓
Need positive correction
```

So:

```text
Rows 1,2 → prediction too high → decrease prediction

Rows 3,4 → prediction too low → increase prediction
```

---

# 5. Calculate Hessian

For squared-error loss:

$$h_i=1$$

Therefore:

| Row | Gradient $g$ | Hessian $h$ |
|---|---:|---:|
| 1 | 6 | 1 |
| 2 | 4 | 1 |
| 3 | -4 | 1 |
| 4 | -6 | 1 |

Now you can see the connection:

```text
Actual + Prediction
        ↓
      Loss
        ↓
Gradient + Hessian
        ↓
Build a tree to correct predictions
```

---

# 6. Now XGBoost tries to build a tree

Suppose XGBoost considers this split:

$$X<3$$

That gives:

```text
                 X < 3
                /     \
             Left     Right
             1,2       3,4
```

So:

### Left child

Rows 1 and 2:

$$G_{left}=6+4=10$$

$$H_{left}=1+1=2$$

### Right child

Rows 3 and 4:

$$G_{right}=-4-6=-10$$

$$H_{right}=1+1=2$$

Now we have the **G and H for each leaf**.

This is where the gradient and Hessian become useful.

---

# 7. Calculate the leaf weight

Remember:

$$w^*=-\frac{G}{H+\lambda}$$

Let's temporarily use:

$$\lambda=0$$

### Left leaf

$$w_{left}^*=-\frac{10}{2}=-5$$

### Right leaf

$$w_{right}^*=-\frac{-10}{2}=5$$

So the tree becomes:

```text
                 X < 3
                /     \
             -5         +5
```

The tree isn't predicting the final answer directly.

It is predicting **corrections**.

That's one of the most important ideas in XGBoost.

---

# 8. Apply learning rate

Suppose:

$$\eta=0.5$$

The tree's corrections become:

### Left:

$$0.5(-5)=-2.5$$

### Right:

$$0.5(5)=2.5$$

Now update the predictions:

### Rows 1 and 2

$$16-2.5=13.5$$

### Rows 3 and 4

$$16+2.5=18.5$$

So after **one tree**:

| Row | Actual | Before | Tree correction | New prediction |
|---|---:|---:|---:|---:|
| 1 | 10 | 16 | -2.5 | 13.5 |
| 2 | 12 | 16 | -2.5 | 13.5 |
| 3 | 20 | 16 | +2.5 | 18.5 |
| 4 | 22 | 16 | +2.5 | 18.5 |

The predictions have moved toward the actual values.

---

# 9. Now calculate the new loss

Before:

$$L=52$$

After the first tree:

For rows 1 and 2:

$$\frac{1}{2}(10-13.5)^2=6.125$$

$$\frac{1}{2}(12-13.5)^2=1.125$$

For rows 3 and 4:

$$\frac{1}{2}(20-18.5)^2=1.125$$

$$\frac{1}{2}(22-18.5)^2=6.125$$

Total:

$$L=6.125+1.125+1.125+6.125=14.5$$

Look at the improvement:

```text
Before tree: 52
After tree:  14.5
```

The tree significantly reduced the loss.

---

# 10. Now the most important connection

You can now see the **entire XGBoost mechanism**:

```text
                Training data
                     ↓
             Initial prediction
                     ↓
                  Loss
                     ↓
          ┌──────────┴──────────┐
          ↓                     ↓
      Gradient               Hessian
          ↓                     ↓
          └──────────┬──────────┘
                     ↓
             Find useful split
                     ↓
          Group samples into leaves
                     ↓
              Sum gradients
                  G = Σg
                     +
              Sum Hessians
                  H = Σh
                     ↓
          Calculate leaf weight
             w* = -G/(H+λ)
                     ↓
              Learning rate
                     ↓
           Tree correction
                     ↓
          Update predictions
                     ↓
               New loss
                     ↓
              Build next tree
```

And **this is why it's called Gradient Boosting**:

Each new tree is built to correct the errors left by the previous model.

---

# 11. Where does `min_child_weight` fit?

Now you can finally see where that hyperparameter belongs.

When XGBoost considers a split, it checks the Hessian of each potential child.

For our left child:

$$H_{left}=2$$

For our right child:

$$H_{right}=2$$

Suppose:

```python
min_child_weight = 3
```

Then:

$$2<3$$

for both children.

Therefore, **XGBoost would reject this split**.

So `min_child_weight` acts **during tree construction**, before accepting the split.

---

# 12. Where does regularization fit?

Suppose:

$$\lambda=2$$

Then our left leaf becomes:

$$w_{left}^*=-\frac{10}{2+2}=-2.5$$

Instead of:

$$w_{left}^*=-5$$

Similarly:

$$w_{right}^*=-\frac{-10}{2+2}=2.5$$

So $\lambda$ makes the correction smaller.

That's **L2 regularization**.

---

# 13. Where does `gamma` fit?

XGBoost doesn't just ask:

> "Can I make this split?"

It also asks:

> **"Is this split actually worth making after considering the improvement and the complexity penalty?"**

That's where $\gamma$ comes in.

Recall:

$$\Omega(f_t)=\gamma T+\frac{1}{2}\lambda\sum_{j=1}^{T}w_j^2$$

So:

- $\gamma$ penalizes having more leaves.
- $\lambda$ penalizes large leaf weights.

Thus:

```text
Candidate split
      ↓
Does each child have enough Hessian?
      ↓
min_child_weight
      ↓
Calculate leaf weights
      ↓
G, H, λ
      ↓
Evaluate objective / gain
      ↓
γ penalizes unnecessary splitting
      ↓
Accept or reject split
```

---

# 14. And then XGBoost repeats everything

After Tree 1:

```text
Predictions:
[13.5, 13.5, 18.5, 18.5]
```

Now these become the predictions used to calculate the **new gradients and Hessians**.

Then:

```text
Tree 2
   ↓
correct Tree 1's remaining errors

Tree 3
   ↓
correct remaining errors

Tree 4
   ↓
correct remaining errors

...
```

Finally:

$$\hat{y}^{(M)}=\hat{y}^{(0)}+\eta\sum_{m=1}^{M}f_m(X)$$

So the final prediction is basically:

> **Initial prediction + correction from Tree 1 + correction from Tree 2 + correction from Tree 3 + ...**

That is the complete connection between **loss → gradient → Hessian → split → leaf weight → regularization → learning rate → prediction → next tree**.